# OCR: V2 

V2 incorporates figures into the md file, saves the figure crops and teh raw html file for later development if needed. Artefacts are saved in data/processed/ where each paper has its own subfolder. 

In [1]:
import os
from dotenv import load_dotenv
_ = load_dotenv(override=True)

In [2]:
# Pointe le client chandra vers notre serveur vLLM (nom DNS sur scirex-net).
# Doit être fait AVANT d'importer chandra.model.vllm car settings est chargé à l'import.
os.environ["VLLM_API_BASE"] = "http://chandra-vllm:8000/v1"
os.environ["VLLM_MODEL_NAME"] = "chandra"

In [3]:
from PIL import Image
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Any
import duckdb
from pdf2image import convert_from_path
import os

from bs4 import BeautifulSoup
import re
import six

from chandra.model.vllm import generate_vllm
from chandra.model.schema import BatchInputItem
from chandra.output import parse_chunks, get_image_name, Markdownify
from IPython.display import Markdown, display

import time
import pymupdf
from PIL import Image
import io

# Scripting my OCR
Development of the V2 of run_ocr.py to keep figure images, add them in the md files and raw html for further analyses (if needed). 

In [ ]:
def pdf_to_images(pdf_path, max_pages = None, dpi = 192):
    '''Take a pdf, return a list of pages converted in png images (RGB)'''
    
    # Open the document and count the number of pages
    doc = pymupdf.open(pdf_path)

    if max_pages is None:
        max_pages = doc.page_count
    else:
        max_pages = min(max_pages, doc.page_count)

    img_list = []
    for page in range(max_pages):
        pix = doc[page].get_pixmap(dpi=dpi)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        img_list.append(img)
    doc.close()
    
    return img_list


In [6]:
def ocr_images(page_images, max_output_tokens=8192, max_workers=8):
    '''Receives a list of PIL images, returns the concatenated markdown and per-page stats, html and images from the article.'''
    
    # Create a batch of images to send to vLLM 
    batch = [BatchInputItem(image=img, prompt_type="ocr_layout") for img in page_images]

    #vLLM OCR
    results = generate_vllm(
        batch,
        max_output_tokens=max_output_tokens,
        max_workers=max_workers,
    )

    per_page_md = []
    per_page_stats = []
    per_page_figures = []
    per_page_html = []

    for i, (result, page_img) in enumerate(zip(results, page_images, strict = True)):
        # Get the markdown, chunks and images for the page
        md = parse_markdown(result.raw)
        chunks = parse_chunks(result.raw, page_img)
        figures = extract_images(result.raw, chunks, page_img)


        per_page_md.append(md)
        per_page_html.append(result.raw)
        per_page_figures.append(figures)
        per_page_stats.append({
            'page': i,
            'tokens': result.token_count,
            'n_chars': len(md),
            'n_images' : len(figures),
            'error': result.error,
        })
    # Aggregate the per-page results into a single html for the entire document
    # "" is to instanciate an empty string, then we concatenate the per-page html with a page separator comment    
    full_html=""
    for i, html in enumerate(per_page_html):
        full_html += f"\n\n<!-- ===== Page {i+1} ===== -->\n\n{html}\n"

    # Aggregate the per-page results into a single md for the entire document    
    full_md = ""
    for i, md in enumerate(per_page_md):
        full_md += f"\n\n<!-- ===== Page {i+1} ===== -->\n\n{md}\n"

    return full_md, per_page_stats, per_page_figures, full_html

In [7]:
def make_paper_dir(arxiv_id: str) -> Path:
    '''Creates the output path for each paper in data/processed as a data lake'''
    
    paper_dir = Path("data/processed") / arxiv_id
    paper_dir.mkdir(parents=True, exist_ok=True)
    return paper_dir

In [8]:
def atomic_write_text(path: Path, text: str) -> None:
    '''A function to write text to a file in an atomic way = to avoid partial writes.'''
    tmp_path = path.parent / (path.name + ".tmp") 
    tmp_path.write_text(text, encoding="utf-8")
    os.replace(tmp_path, path)

def atomic_write_image(path: Path, image: Image.Image) -> None:
    '''A function to write an image to a file in an atomic way = to avoid partial writes.'''
    tmp_path = path.parent / (path.name + ".tmp")
    image.save(tmp_path, format="WEBP")           
    os.replace(tmp_path, path)

In [9]:
def save_all(paper_dir: Path, html, md, stats, images):
    '''Writes all artefacts from the OCR process to disk.'''

    # html:
    atomic_write_text(paper_dir / f"{arxiv_id}.html", html)

    # figures:
    for figures in images:
        for name, img in figures.items():
            atomic_write_image(paper_dir / name, img)

    # md:
    atomic_write_text(paper_dir / f"{arxiv_id}.md", md)

In [10]:
# Modifications of the original functions in chandra to return diagram as figures: (issue found on 2401.01624.pdf)
# Vendored from chandra/output.py v0.2.0 — modifs : IMAGE_LABELS tunable

# Modify this list if you want to include other types of images (e.g., chemical formulas) as figures. 
# The original code only included "Image" and "Figure" labels. 
IMAGE_LABELS = ("Image", "Figure", "Diagram") 

# Modification1
def parse_html(html: str, include_headers_footers: bool = False, include_images: bool = True):
    soup = BeautifulSoup(html, "html.parser")
    top_level_divs = soup.find_all("div", recursive=False)
    out_html = ""
    image_idx = 0
    div_idx = 0
    for div in top_level_divs:
        div_idx += 1
        label = div.get("data-label")

        if label == "Blank-Page":
            continue

        # Skip headers and footers if not included
        if label and not include_headers_footers:
            if label in ["Page-Header", "Page-Footer"]:
                continue
        if label and not include_images:
            if label in IMAGE_LABELS:
                continue

        if label in IMAGE_LABELS:
            img = div.find("img")
            img_src = get_image_name(html, div_idx)

            # If no tag, add one in
            if img:
                img["src"] = img_src
                image_idx += 1
            else:
                img = BeautifulSoup(f"<img src='{img_src}'/>", "html.parser")
                div.append(img)

        # Strip img tags without src in non-image blocks (model hallucinations)
        if label not in IMAGE_LABELS:
            for img_tag in div.find_all("img"):
                if not img_tag.get("src"):
                    img_tag.decompose()

        # Wrap text content in <p> tags if no inner HTML tags exist
        if label in ["Text"] and not re.search(
            "<.+>", str(div.decode_contents()).strip()
        ):
            # Add inner p tags if missing for text blocks
            text_content = str(div.decode_contents()).strip()
            text_content = f"<p>{text_content}</p>"
            div.clear()
            div.append(BeautifulSoup(text_content, "html.parser"))

        content = str(div.decode_contents())
        out_html += content
    return out_html

# Modification 2:
def extract_images(html: str, chunks: dict, image: Image.Image):
    images = {}
    div_idx = 0
    for idx, chunk in enumerate(chunks):
        div_idx += 1
        if chunk["label"] in IMAGE_LABELS:
            img = BeautifulSoup(chunk["content"], "html.parser").find("img")
            if not img:
                continue
            bbox = chunk["bbox"]
            try:
                block_image = image.crop(bbox)
            except ValueError:
                # Happens when bbox coordinates are invalid
                continue
            img_name = get_image_name(html, div_idx)
            images[img_name] = block_image
    return images


In [14]:
# Verification of the pipeline on a single paper
arxiv_id = "2401.13610"
pdf_path = f"data/raw/pdfs/{arxiv_id}.pdf"

img_list = pdf_to_images(pdf_path, max_pages = 4, dpi=192)
md, stats, figures, html = ocr_images(img_list)
dir = make_paper_dir(arxiv_id)
save_all(dir, html, md, stats, figures)

In [ ]:
# Test to split the html file in pages:
import re

PAGE_MARKER = re.compile(r"<!-- ===== Page \d+ ===== -->")

def split_pages(html: str) -> list[str]:
    pages = PAGE_MARKER.split(html)
    return [p.strip() for p in pages[1:]]   # [0] Get rid of the first element because it is the first page marker


html = Path(f"data/processed/2401.01623/2401.01623.html").read_text(encoding="utf-8")
pages = split_pages(html)
pages[1]

' Page 1 ===== -->\n\n<div data-bbox="21 275 57 707" data-label="Page-Header">arXiv:2401.01623v4 [cs.AI] 25 Jan 2024</div>\n<div data-bbox="297 155 699 175" data-label="Section-Header">\n<h1>Can AI Be as Creative as Humans?</h1>\n</div>\n<div data-bbox="151 196 846 231" data-label="Text">\n<p>Haonan Wang<sup>1</sup> James Zou<sup>2</sup> Michael Mozer<sup>3</sup> Anirudh Goyal<sup>3</sup> Alex Lamb<sup>4</sup> Linjun Zhang<sup>5</sup><br/>\n  Weijie J. Su<sup>6</sup> Zhun Deng<sup>7</sup> Michael Qizhe Xie<sup>1</sup> Hannah Brown<sup>1</sup> Kenji Kawaguchi<sup>1</sup></p>\n</div>\n<div data-bbox="263 237 734 253" data-label="Text">\n<p><sup>1</sup>National University of Singapore <sup>2</sup>Stanford University <sup>3</sup>Google DeepMind</p>\n</div>\n<div data-bbox="207 255 790 271" data-label="Text">\n<p><sup>4</sup>Microsoft Research <sup>5</sup>Rutgers University <sup>6</sup>University of Pennsylvania <sup>7</sup>Columbia University</p>\n</div>\n<div data-bbox="304 281 693 296" d

# V3:

In [4]:
def pdf_to_images(pdf_path, max_pages = None, dpi = 192):
    '''Take a pdf, return a list of pages converted in png images'''
    
    # Open the document and count the number of pages
    doc = pymupdf.open(pdf_path)

    if max_pages is None:
        max_pages = doc.page_count
    else:
        max_pages = min(max_pages, doc.page_count)

    img_list = []
    for page in range(max_pages):
        pix = doc[page].get_pixmap(dpi=dpi)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        img_list.append(img)
    doc.close()
    
    return img_list


In [ ]:
# Vendored from chandra/output.py v0.2.0 — modifs : IMAGE_LABELS tunable
IMAGE_LABELS = ("Image", "Figure", "Diagram")

In [ ]:
# Vendored from chandra/output.py v0.2.0 — modifs : IMAGE_LABELS tunable

# Modification 1: parse_html now takes chunks as input instead of the raw html string
def parse_html(
    html: str,
    chunks: list[dict],
    include_headers_footers: bool = False,
    include_images: bool = True,
):
    out_html = ""
    for div_idx, chunk in enumerate(chunks, start=1):
        label = chunk["label"]
        content = chunk["content"]

        if label == "Blank-Page":
            continue

        # Skip headers and footers if not included
        if not include_headers_footers and label in ["Page-Header", "Page-Footer"]:
            continue
        if not include_images and label in IMAGE_LABELS:
            continue

        # Traiter les images
        if label in IMAGE_LABELS:
            img_src = get_image_name(html, div_idx)
            img_soup = BeautifulSoup(content, "html.parser")
            img = img_soup.find("img")
            if img:
                img["src"] = img_src
            else:
                img = BeautifulSoup(f"<img src='{img_src}'/>", "html.parser")
                img_soup.append(img)
            content = str(img_soup)

        # Traiter le texte
        elif label == "Text" and not re.search("<.+>", content.strip()):
            content = f"<p>{content.strip()}</p>"

        out_html += content
    return out_html

In [ ]:
# Vendored from chandra/output.py v0.2.0 — modifs : IMAGE_LABELS tunable

# Modification 2: extract_images now takes chunks as input instead of the raw html string
def extract_images(html: str, chunks: list[dict], image: Image.Image):
    images = {}
    for div_idx, chunk in enumerate(chunks, start=1):
        if chunk["label"] in IMAGE_LABELS:
            img = BeautifulSoup(chunk["content"], "html.parser").find("img")
            if not img:
                continue
            bbox = chunk["bbox"]
            try:
                block_image = image.crop(bbox)
            except ValueError:
                continue
            img_name = get_image_name(html, div_idx)
            images[img_name] = block_image
    return images

In [ ]:
# Vendored from chandra/output.py v0.2.0 — modifs : IMAGE_LABELS tunable

# Modification 3: parse_markdown now takes chunks as input instead of the raw html string
def parse_markdown(html: str):
    md_cls = Markdownify(
        heading_style="ATX",
        bullets="-",
        escape_misc=False,
        escape_underscores=True,
        escape_asterisks=True,
        escape_dollars=True,
        sub_symbol="<sub>",
        sup_symbol="<sup>",
        inline_math_delimiters=("$", "$"),
        block_math_delimiters=("$$", "$$"),
    )
    try:
        markdown = md_cls.convert(html)
    except Exception as e:
        print(f"Error converting HTML to Markdown: {e}")
        markdown = ""
    return markdown.strip()

In [ ]:
def ocr_images(page_images, max_output_tokens=8192, max_workers=8):
    '''Receives a list of PIL images, returns the concatenated markdown and per-page stats, html and images from the article.'''
    
    # Create a batch of images to send to vLLM 
    batch = [BatchInputItem(image=img, prompt_type="ocr_layout") for img in page_images]

    #vLLM OCR
    results = generate_vllm(
        batch,
        max_output_tokens=max_output_tokens,
        max_workers=max_workers,
    )

    per_page_md = []
    per_page_stats = []
    per_page_figures = []
    per_page_html = []

    for i, (result, page_img) in enumerate(zip(results, page_images, strict = True)):
        # Get the markdown, chunks and images for the page
        chunks = parse_chunks(result.raw, page_img)
        figures = extract_images(result.raw, chunks, page_img)
        html = parse_html(result.raw, chunks)  # Passe les chunks ici
        md = parse_markdown(html)  # Utilise le HTML déjà généré

        per_page_md.append(md)
        per_page_html.append(result.raw)
        per_page_figures.append(figures)
        per_page_stats.append({
            'page': i,
            'n_tokens': result.token_count,
            'n_chars': len(md),
            'n_images': len(figures),
            'error': result.error,
        })
        
    # Aggregate the per-page results into a single html for the entire document
    # "" is to instanciate an empty string, then we concatenate the per-page html with a page separator comment
    full_html = "\n\n".join(
        f"<!-- ===== Page {i+1} ===== -->\n\n{html}"
        for i, html in enumerate(per_page_html)
    )

    # Aggregate the per-page results into a single md for the entire document    
    full_md = "\n\n".join(
        f"<!-- ===== Page {i+1} ===== -->\n\n{md}"
        for i, md in enumerate(per_page_md)
    )
    return full_md, per_page_stats, per_page_figures, full_html

In [10]:
def make_paper_dir(arxiv_id: str) -> Path:
    '''Creates the output path for each paper in data/processed as a data lake'''
    
    paper_dir = Path("data/processed") / arxiv_id
    paper_dir.mkdir(parents=True, exist_ok=True)
    return paper_dir

In [11]:
def atomic_write_text(path: Path, text: str) -> None:
    '''A function to write text to a file in an atomic way = to avoid partial writes.'''
    tmp_path = path.parent / (path.name + ".tmp") 
    tmp_path.write_text(text, encoding="utf-8")
    os.replace(tmp_path, path)

def atomic_write_image(path: Path, image: Image.Image) -> None:
    '''A function to write an image to a file in an atomic way = to avoid partial writes.'''
    tmp_path = path.parent / (path.name + ".tmp")
    image.save(tmp_path, format="WEBP")           
    os.replace(tmp_path, path)

In [ ]:
def save_all(paper_dir: Path, html, md, stats, images):
    '''Writes all artefacts from the OCR process to disk.'''

    # html:
    atomic_write_text(paper_dir / f"{arxiv_id}.html", html)

    # figures:
    for figures in images:
        for name, img in figures.items():
            atomic_write_image(paper_dir / name, img)

    # md:
    atomic_write_text(paper_dir / f"{arxiv_id}.md", md)

    # update the paper_local table: 
    nb_pages_pdf = len(img_list)
    nb_pages_ocr = len(per_page_md)
    keyword_for_ocr = "args.table"
    


    # update the ocr_stats table:
    for stat in stats:
        conn.execute(
            """
            INSERT INTO ocr_stats (arxiv_id, page, n_tokens, n_chars, n_images, error)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (arxiv_id, stat['page'], stat['n_tokens'], stat['n_chars'], stat['n_images'], stat['error'])
        )

In [14]:
# Verification of the pipeline on a single paper
arxiv_id = "2401.13610"
pdf_path = f"data/raw/pdfs/{arxiv_id}.pdf"

img_list = pdf_to_images(pdf_path, max_pages = 4, dpi=192)
md, stats, figures, html = ocr_images(img_list)
dir = make_paper_dir(arxiv_id)
save_all(dir, html, md, stats, figures)

# Orchestration

The input given to the run_ocr.py script in the V2 must be a table from my database of articles of interests where pdfs have been already downloaded and ready for OCR. 

The output should be what has been developped above and an update of the table to reflect at all time what has been done to the papers to ensure idempotency. 